# Conservative 2D regrid — curvilinear target

**Conservative regridding** resamples a gridded field while preserving its
area-weighted integral: each output cell is the area-weighted average of
the source cells it overlaps. It's the right tool for fluxes and intensive
quantities — precipitation, sea-surface temperature, mass-balance budgets —
where bilinear or nearest-neighbor interpolation would bias the total.

The fast `.regrid.conservative` accessor only works on **1D-separable**
grids: plain rectilinear lat/lon, where lat depends only on `y` and lon
only on `x`. **Curvilinear** grids store coordinates as 2D arrays
`lat(y, x)` / `lon(y, x)` and are common in ocean models (ORCA, tripolar)
or any rotated/projected setup. Their cells aren't axis-aligned, so we
drop to `.regrid.conservative_2d`, which builds the full 2D polygon
intersection.

**In this notebook.** We regrid a smooth analytic two-bump field from a
regular lat/lon grid onto a 30°-rotated curvilinear target — a minimal
stand-in for any curvilinear ocean grid — and verify that the
area-weighted integral is preserved to machine precision.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import xarray_regrid  # noqa: F401
from xarray_regrid import ConservativeRegridder

## Source — regular 1° lat/lon, analytic two-bump field

A simple synthetic field — one positive Gaussian bump in the northern
hemisphere, one negative in the southern — gives us something we can
visually track from source to target.

In [ ]:
lat = np.linspace(-60, 60, 121)
lon = np.linspace(-120, 120, 241)
Lo, La = np.meshgrid(lon, lat)
field = (
    np.exp(-((Lo - 40) ** 2 + (La - 20) ** 2) / 500)
    - np.exp(-((Lo + 60) ** 2 + (La + 15) ** 2) / 400)
)
src = xr.DataArray(
    field,
    dims=("latitude", "longitude"),
    coords={"latitude": lat, "longitude": lon},
)
src.plot(figsize=(8, 3.5), cmap="RdBu_r", center=0)
plt.title("source: analytic two-bump field")
plt.tight_layout()

## Target — rotated curvilinear grid

To break 1D-separability we take a regular `(ny, nx)` mesh and rotate it
30° in the lat/lon plane. The result has the same kind of 2D coordinate
variables you'd find in an ORCA or rotated-pole grid: `longitude(ny, nx)`
and `latitude(ny, nx)` riding on a non-axis-aligned mesh.

In [ ]:
ny, nx = 30, 50
xi, yi = np.meshgrid(
    np.linspace(-110, 110, nx),
    np.linspace(-45, 45, ny),
    indexing="xy",
)
th = np.deg2rad(30)
lon2d = xi * np.cos(th) - yi * np.sin(th)
lat2d = xi * np.sin(th) + yi * np.cos(th)
target = xr.Dataset(coords={
    "longitude": (("ny", "nx"), lon2d),
    "latitude":  (("ny", "nx"), lat2d),
})

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lon2d, lat2d, color="0.3", lw=0.4)
ax.plot(lon2d.T, lat2d.T, color="0.3", lw=0.4)
ax.set_title("curvilinear target (30° rotation)")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_aspect("equal")

## Regrid and plot

`ConservativeRegridder` builds the weight matrix once — that's the
expensive step, since it computes the area of every source/target polygon
intersection — and lets us reuse it for the apply step and for the
diagnostic in the next cell. The one-shot equivalent is
`src.regrid.conservative_2d(target, ...)`. Either way, output is on the
curvilinear `(ny, nx)` mesh, with NaN where target cells fall outside the
source domain.

In [ ]:
rgr = ConservativeRegridder(
    src, target, x_coord="longitude", y_coord="latitude",
)
regridded = rgr.regrid(src)

fig, ax = plt.subplots(figsize=(8, 4))
pc = ax.pcolormesh(lon2d, lat2d, regridded.values, cmap="RdBu_r",
                   shading="auto", vmin=-1, vmax=1)
fig.colorbar(pc, ax=ax, shrink=0.8)
ax.set_title("regridded onto rotated curvilinear grid")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_aspect("equal")

## Conservation check

The regridder stores its area-intersection matrix
`A[i, j] = area(target_i ∩ source_j)`. For target cells fully inside the
source domain, the area-weighted sum of outputs equals the direct A·s
integral to machine precision — the defining property of *conservative*
regridding.

In [ ]:
A = rgr.areas
src_cover = np.ravel(A.sum(axis=0).todense())
tgt_cover = A.sum(axis=1).todense().reshape(regridded.shape)
valid = np.isfinite(regridded.values)

direct = float((src.values.ravel() * src_cover).sum())
via_regrid = float((regridded.values[valid] * tgt_cover[valid]).sum())
print(f"direct        : {direct:.6f}")
print(f"via regrid    : {via_regrid:.6f}")
print(f"relative err  : {abs(direct - via_regrid) / max(abs(direct), 1e-12):.2e}")
print(f"coverage      : {valid.mean():.2%} of target cells inside source")